# 오토인코더로 이미지의 특징을 추출하기

In [ ]:
import torch # 파이토치 임포트
import torchvision # 토치비전 임포트
import torch.nn.functional as F # 신경망 함수 임포트
from torch import nn, optim # 모델 및 최적화 임포트
from torchvision import transforms, datasets # 데이터 변환 및 데이터셋 임포트

import matplotlib.pyplot as plt # 시각화 라이브러리
from mpl_toolkits.mplot3d import Axes3D # 3D 시각화 도구
from matplotlib import cm # 컬러맵 설정
import numpy as np # 수치 계산 라이브러리
import os # 운영체제 제어

In [ ]:
# 하이퍼파라미터
EPOCH = 10 # 학습 반복 횟수
BATCH_SIZE = 64 # 배치 크기
USE_CUDA = torch.cuda.is_available() # GPU 확인
DEVICE = torch.device("cuda" if USE_CUDA else "cpu") # 장치 설정
print("Using Device:", DEVICE) # 장치 출력

In [ ]:
# Fashion MNIST 데이터셋
trainset = datasets.FashionMNIST(
    root      = './.data/', 
    train     = True, # 학습용
    download  = True, 
    transform = transforms.ToTensor() # 텐서 변환
)
train_loader = torch.utils.data.DataLoader(
    dataset     = trainset,
    batch_size  = BATCH_SIZE,
    shuffle     = True, # 섞기
    num_workers = 0  # 병렬 처리
)

In [ ]:
class Autoencoder(nn.Module): # 오토인코더 클래스
    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential( # 인코더 정의
            nn.Linear(28*28, 128), # 784 -> 128
            nn.ReLU(),
            nn.Linear(128, 64), # 128 -> 64
            nn.ReLU(),
            nn.Linear(64, 12), # 64 -> 12
            nn.ReLU(),
            nn.Linear(12, 3),   # 특징 압축 (3차원)
        )
        self.decoder = nn.Sequential( # 디코더 정의
            nn.Linear(3, 12), # 3 -> 12
            nn.ReLU(),
            nn.Linear(12, 64), # 12 -> 64
            nn.ReLU(),
            nn.Linear(64, 128), # 64 -> 128
            nn.ReLU(),
            nn.Linear(128, 28*28), # 128 -> 784
            nn.Sigmoid(),       # 0~1 출력
        )

    def forward(self, x): # 순전파
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [ ]:
def train(autoencoder, train_loader): # 학습 함수
    autoencoder.train() # 학습 모드
    for step, (x, label) in enumerate(train_loader):
        x = x.view(-1, 28*28).to(DEVICE) # 평탄화
        y = x.view(-1, 28*28).to(DEVICE) # 정답 설정
        label = label.to(DEVICE)

        encoded, decoded = autoencoder(x) # 실행

        loss = criterion(decoded, y) # 손실 계산
        optimizer.zero_grad() # 기울기 초기화
        loss.backward() # 역전파
        optimizer.step() # 업데이트

In [ ]:
autoencoder = Autoencoder().to(DEVICE) # 모델 생성
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.005) # 최적화 설정
criterion = nn.MSELoss() # 손실 함수 설정

# 원본 이미지를 시각화 하기 (첫번째 열)
view_data = trainset.data[:5].view(-1, 28*28)
view_data = view_data.type(torch.FloatTensor)/255. # 정규화

# 결과를 저장할 디렉토리 생성
if not os.path.exists('results'): # 폴더 확인
    os.makedirs('results')

In [ ]:
for epoch in range(1, EPOCH+1): # 에포크 반복
    train(autoencoder, train_loader) # 학습

    # 디코더에서 나온 이미지를 시각화 하기 (두번째 열)
    test_x = view_data.to(DEVICE)
    _, decoded_data = autoencoder(test_x) # 복원

    # 원본과 디코딩 결과 비교해보기
    f, a = plt.subplots(2, 5, figsize=(5, 2)) # 그래프 생성
    print("[Epoch {}]".format(epoch))
    for i in range(5): # 원본 출력
        img = np.reshape(view_data.data.numpy()[i],(28, 28))
        a[0][i].imshow(img, cmap='gray')
        a[0][i].set_xticks(()); a[0][i].set_yticks(())

    for i in range(5): # 복원 출력
        img = np.reshape(decoded_data.to("cpu").data.numpy()[i], (28, 28))
        a[1][i].imshow(img, cmap='gray')
        a[1][i].set_xticks(()); a[1][i].set_yticks(())
    
    # 학습 과정 이미지 저장
    plt.savefig(f'results/epoch_{epoch:02d}.png', dpi=300, bbox_inches='tight')
    plt.show() # 출력

# 잠재변수 들여다보기

In [ ]:
# 잠재변수를 3D 플롯으로 시각화
view_data = trainset.data[:200].view(-1, 28*28) # 샘플 200개
view_data = view_data.type(torch.FloatTensor)/255. # 정규화
test_x = view_data.to(DEVICE)
encoded_data, _ = autoencoder(test_x) # 특징 추출
encoded_data = encoded_data.to("cpu") # 이동

In [ ]:
CLASSES = { # 클래스 정의
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

# 새로운 figure 생성
plt.clf()  # 초기화
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')  # 3D 생성

X = encoded_data.data[:, 0].numpy() # X축
Y = encoded_data.data[:, 1].numpy() # Y축
Z = encoded_data.data[:, 2].numpy() # Z축

labels = trainset.targets[:200].numpy() # 레이블

# 산점도 그리기
scatter = ax.scatter(X, Y, Z, c=labels, cmap='rainbow', s=100, alpha=0.8)

# 텍스트 레이블 추가 (일부만 표시)
for i, (x, y, z, s) in enumerate(zip(X, Y, Z, labels)):
    if i % 20 == 0:  # 일부 표시
        name = CLASSES[s]
        color = cm.rainbow(int(255*s/9))
        ax.text(x, y, z, name, color='black', fontsize=10, 
                bbox=dict(facecolor=color, alpha=0.5, edgecolor='black', pad=2))

# 축 레이블 추가
ax.set_xlabel('Latent 1', fontsize=12)
ax.set_ylabel('Latent 2', fontsize=12)
ax.set_zlabel('Latent 3', fontsize=12)

# 축 범위 설정
margin = 0.1 
x_range = X.max() - X.min()
y_range = Y.max() - Y.min()
z_range = Z.max() - Z.min()

ax.set_xlim(X.min() - margin * x_range, X.max() + margin * x_range)
ax.set_ylim(Y.min() - margin * y_range, Y.max() + margin * y_range)
ax.set_zlim(Z.min() - margin * z_range, Z.max() + margin * z_range)

# 컬러바 추가
cbar = plt.colorbar(scatter)
cbar.set_label('Class', fontsize=12)

# 제목 추가
plt.title('3D Visualization of Latent Space', fontsize=14, pad=20)

# 그리드 추가
ax.grid(True)

# 시점 설정
ax.view_init(elev=20, azim=45)

# 레이아웃 조정
plt.tight_layout()

# 저장 및 표시
plt.savefig('results/latent_space_3d.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n모든 이미지가 'results' 디렉토리에 저장되었습니다.")
print("- 학습 과정 이미지: epoch_01.png ~ epoch_10.png")
print("- 3D 시각화 이미지: latent_space_3d.png")